[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C72_MultiView_Geometry_Course/02_calibration/02_calibration.ipynb)

# 02 · 标定：内参、外参与验收

四件事：

1. **验证 Zhang 的约束计数**：1 视角 rank 2、2 视角 rank 4、**3 视角 rank 5（刚好可解）**；
   而「纯平移」和「只绕光轴转」拍多少张都是 **rank 2**。
2. **把归一化的收益诚实地量一遍** —— 结果与教科书的常见说法不完全一样。
3. **从多视角单应解出 K**（闭式解，相对误差 5e-15）。
4. **可辨识性实验**：σ(pitch) 从 **3.74° 降到 0.035°**，
   靠的不是多拍，而是**扩大图像行覆盖**。

## 0 · 环境与真值

In [ ]:
import numpy as np

print('numpy', np.__version__)

F, CX, CY = 1200.0, 960.0, 540.0
W, HGT = 1920, 1080
H_CAM = 1.5
K_TRUE = np.array([[F, 0, CX], [0, F, CY], [0, 0, 1.]])

def rot(rx, ry, rz):
    '''按 X-Y-Z 顺序的欧拉角（度）构造旋转矩阵。'''
    rx, ry, rz = map(np.deg2rad, (rx, ry, rz))
    Rx = np.array([[1,0,0],[0,np.cos(rx),-np.sin(rx)],[0,np.sin(rx),np.cos(rx)]])
    Ry = np.array([[np.cos(ry),0,np.sin(ry)],[0,1,0],[-np.sin(ry),0,np.cos(ry)]])
    Rz = np.array([[np.cos(rz),-np.sin(rz),0],[np.sin(rz),np.cos(rz),0],[0,0,1]])
    return Rz @ Ry @ Rx

# 棋盘格：9x6 个内角点，格距 30mm，位于板坐标系的 Z=0 平面
NX, NY, PITCH_M = 9, 6, 0.030
gx, gy = np.meshgrid(np.arange(NX) * PITCH_M, np.arange(NY) * PITCH_M)
BOARD = np.stack([gx.ravel(), gy.ravel()], axis=1)      # (54, 2)
print(f'标定板 {NX}x{NY} 内角点，格距 {PITCH_M*1000:.0f}mm，'
      f'板面 {(NX-1)*PITCH_M*1000:.0f}x{(NY-1)*PITCH_M*1000:.0f}mm')

## 1 · 多视角与单应

板在自己坐标系里是 $Z=0$，所以「板 → 图像」是一个单应 $H=K[r_1\;r_2\;t]$。

In [ ]:
def H_of_view(R, t, Kmat=None):
    '''平面板（Z=0）到图像的单应。'''
    Kmat = K_TRUE if Kmat is None else Kmat
    Hm = Kmat @ np.column_stack([R[:, 0], R[:, 1], np.asarray(t, float)])
    return Hm / Hm[2, 2]

def apply_H(Hm, pts):
    ph = np.column_stack([pts, np.ones(len(pts))]) @ Hm.T
    return ph[:, :2] / ph[:, 2:]

VIEWS_GOOD = [
    (rot(  0,   0,   0), [ 0.00,  0.00, 0.50]),
    (rot( 25, -20,  10), [ 0.05,  0.00, 0.55]),
    (rot(-30,  15,  -8), [-0.04,  0.02, 0.60]),
    (rot( 10,  35,  20), [ 0.02, -0.03, 0.52]),
    (rot(-18, -28, -15), [ 0.01,  0.04, 0.58]),
]

# 另外四个视角：把板推到画幅的四个象限（板距 1.0m，这样整块板仍在画幅内）
VIEWS_EDGE = [
    (rot( 18, -24,  -8), [-0.58, -0.335, 1.0]),
    (rot( 16,  25,   8), [ 0.34, -0.335, 1.0]),
    (rot(-20,  22,   8), [ 0.34,  0.185, 1.0]),
    (rot(-17, -21,  -8), [-0.58,  0.185, 1.0]),
]
VIEWS_FULL = VIEWS_GOOD + VIEWS_EDGE

def corners_of(views):
    return np.vstack([apply_H(H_of_view(R, t), BOARD) for R, t in views])

RD = np.hypot(CX, CY)
print(f"{'配置':22s} {'角点数':>6s} {'画幅内':>7s} {'外 1/3 半径的点数':>18s} {'Δv':>7s}")
for name, vs in [('VIEWS_GOOD (5)', VIEWS_GOOD), ('VIEWS_FULL (9)', VIEWS_FULL)]:
    uv = corners_of(vs)
    rn = np.linalg.norm(uv - np.array([CX, CY]), axis=1) / RD
    ins = ((uv[:,0] >= 0) & (uv[:,0] < W) & (uv[:,1] >= 0) & (uv[:,1] < HGT))
    print(f'{name:22s} {len(uv):6d} {ins.mean():7.2f} {int((rn>=2/3).sum()):18d} '
          f'{np.ptp(uv[:,1]):7.0f}')
    assert ins.all(), f'{name}: 所有角点都应在画幅内'

UV_ALL_GOOD = corners_of(VIEWS_GOOD)
UV_ALL_FULL = corners_of(VIEWS_FULL)
n_outer_good = int((np.linalg.norm(UV_ALL_GOOD - [CX,CY], axis=1) / RD >= 2/3).sum())
n_outer_full = int((np.linalg.norm(UV_ALL_FULL - [CX,CY], axis=1) / RD >= 2/3).sum())
assert n_outer_good <= 2, '五视角配置几乎没有边缘角点'
assert n_outer_full > 30, '加了四象限视角后边缘才被覆盖'
print(f'\n✅ 两个配置的 rank 都会是 5、Δv 都够，'
      f'但边缘角点数是 **{n_outer_good} vs {n_outer_full}**')
print('   → **「朝向够」与「覆盖到边缘」是两件不同的事**'
      '（模块 01 第 4 节的陷阱就在这里）')

## 2 · Zhang 的约束计数与退化配置

$\omega=K^{-\top}K^{-1}$ 对称齐次 → **5 个自由度**；每视角给 **2 个**线性约束。
所以至少要 3 个视角 —— 但**「视角」指的是朝向不同，不是位置不同**。

In [ ]:
def v_ij(h, i, j):
    '''Zhang 论文里的 v_ij 行向量（作用在 omega 的 6 个分量上）。'''
    return np.array([h[0,i]*h[0,j],
                     h[0,i]*h[1,j] + h[1,i]*h[0,j],
                     h[1,i]*h[1,j],
                     h[2,i]*h[0,j] + h[0,i]*h[2,j],
                     h[2,i]*h[1,j] + h[1,i]*h[2,j],
                     h[2,i]*h[2,j]])

def constraint_matrix(views):
    rows = []
    for R, t in views:
        h = H_of_view(R, t)
        rows.append(v_ij(h, 0, 1))                      # r1 · r2 = 0
        rows.append(v_ij(h, 0, 0) - v_ij(h, 1, 1))      # |r1| = |r2|
    return np.array(rows)

CONFIGS = {
    '1 视角':                VIEWS_GOOD[:1],
    '2 视角（不同朝向）':     VIEWS_GOOD[:2],
    '3 视角（不同朝向）':     VIEWS_GOOD[:3],
    '5 视角（不同朝向）':     VIEWS_GOOD,
    '退化：5 张纯平移':       [(rot(0,0,0), [0.02*i, 0.01*i, 0.50+0.05*i]) for i in range(5)],
    '退化：5 张只绕光轴转':   [(rot(0,0,15*i), [0,0,0.50]) for i in range(5)],
    '5 张只绕一个轴倾斜':     [(rot(8*i,0,0), [0,0,0.50]) for i in range(5)],
}
ranks = {}
print(f"{'采集配置':24s} {'A 的形状':>10s} {'rank':>5s}  结论")
for name, vs in CONFIGS.items():
    A = constraint_matrix(vs)
    r = int(np.linalg.matrix_rank(A, tol=1e-8))
    ranks[name] = r
    print(f'{name:24s} {str(A.shape):>10s} {r:5d}  '
          f"{'可解' if r >= 5 else '**欠定**'}")

assert ranks['1 视角'] == 2 and ranks['2 视角（不同朝向）'] == 4
assert ranks['3 视角（不同朝向）'] == 5, '三个不同朝向刚好够'
assert ranks['5 视角（不同朝向）'] == 5, 'rank 在 5 就饱和'
assert ranks['退化：5 张纯平移'] == 2, '纯平移拍 5 张 == 拍 1 张'
assert ranks['退化：5 张只绕光轴转'] == 2
assert ranks['5 张只绕一个轴倾斜'] == 5, '只倾斜一个轴就够'
print('\n✅ rank 在 3 个不同朝向时到 5 并饱和；'
      '而两种退化配置**拍 5 张仍然只有 rank 2**')
print('   → **多拍不能替代换朝向**，而 rank 检查只要一次 SVD')

## 3 · DLT 与归一化：一个诚实的测量

常见说法是「归一化换来几个数量级的精度」。**下面把它量一遍。**

In [ ]:
def normalize_pts(p):
    '''平移到质心、缩放到平均距离 sqrt(2)。返回 (归一化点, 变换矩阵 T)。'''
    c = p.mean(0)
    d = np.sqrt(((p - c) ** 2).sum(1)).mean()
    s = np.sqrt(2) / d
    T = np.array([[s, 0, -s*c[0]], [0, s, -s*c[1]], [0, 0, 1.]])
    ph = np.column_stack([p, np.ones(len(p))]) @ T.T
    return ph[:, :2], T

def dlt_homography(src, dst, normalize=True, dtype=np.float64):
    '''最小二乘解单应。返回 (H, cond(A))。'''
    if normalize:
        s_n, Ts = normalize_pts(src); d_n, Td = normalize_pts(dst)
    else:
        s_n, d_n, Ts, Td = src, dst, np.eye(3), np.eye(3)
    A = []
    for (x, y), (u, v) in zip(s_n, d_n):
        A.append([-x, -y, -1,  0,  0,  0, u*x, u*y, u])
        A.append([ 0,  0,  0, -x, -y, -1, v*x, v*y, v])
    A = np.array(A, dtype=dtype)
    cond = float(np.linalg.cond(A.astype(np.float64)))
    _, _, Vt = np.linalg.svd(A)
    Hn = np.array(Vt[-1].reshape(3, 3), dtype=np.float64)
    Hm = np.linalg.inv(Td) @ Hn @ Ts
    return Hm / Hm[2, 2], cond

R0, t0 = VIEWS_GOOD[1]
H_TRUE = H_of_view(R0, t0)
UV_TRUE = apply_H(H_TRUE, BOARD)
rng = np.random.default_rng(7)

print(f"{'噪声(px)':>9s} {'归一化':>7s} {'cond(A)':>11s} {'H 相对误差':>12s}")
res = {}
for noise in [0.0, 0.2, 1.0]:
    for nz in [False, True]:
        errs, conds = [], []
        for _ in range(30):
            uv = UV_TRUE + (0 if noise == 0 else rng.normal(0, noise, UV_TRUE.shape))
            Hm, c = dlt_homography(BOARD, uv, normalize=nz)
            conds.append(c)
            errs.append(np.linalg.norm(Hm - H_TRUE) / np.linalg.norm(H_TRUE))
        res[(noise, nz)] = (np.mean(conds), np.mean(errs))
        print(f'{noise:9.1f} {str(nz):>7s} {np.mean(conds):11.3e} {np.mean(errs):12.3e}')

# 无噪声：归一化的数值收益是真实且巨大的
gain = res[(0.0, False)][1] / res[(0.0, True)][1]
print(f'\n无噪声时归一化带来的精度提升 = **{gain:.0f} 倍**')
assert gain > 10, f'无噪声时归一化应有显著收益，实测 {gain:.1f}'

# 有噪声：收益被噪声淹没
for noise in [0.2, 1.0]:
    e_no, e_yes = res[(noise, False)][1], res[(noise, True)][1]
    print(f'噪声 {noise} px: 未归一化 {e_no:.2e} vs 归一化 {e_yes:.2e}'
          f'  → 差别 {abs(e_no-e_yes)/max(e_no,e_yes)*100:.0f}%')
    assert abs(e_no - e_yes) / max(e_no, e_yes) < 0.5, \
        '有噪声时两者应当在同一量级'

# 条件数的改善始终是 4-5 个数量级
for noise in [0.2, 1.0]:
    assert res[(noise, False)][0] / res[(noise, True)][0] > 1e3
print('\n✅ 归一化改善的是**数值条件**（4–5 个数量级），'
      '而真实标定的精度由角点噪声支配')
print('   → 它是好习惯（零成本），但**不是精度手段**')

## 3b · 更糟的点分布，以及 float32

如果归一化的收益真的是精度，那它应该在点聚集、近共线、低精度浮点下显现出来。
**实测：都没有。**

In [ ]:
def rel_err(src, uv_true, noise, nz, dtype, trials=30, seed=11):
    r = np.random.default_rng(seed)
    es = []
    for _ in range(trials):
        uv = uv_true + (0 if noise == 0 else r.normal(0, noise, uv_true.shape))
        Hm, _ = dlt_homography(src, uv, normalize=nz, dtype=dtype)
        es.append(np.linalg.norm(Hm - H_TRUE) / np.linalg.norm(H_TRUE))
    return float(np.mean(es))

# 三种点分布：把板坐标压扁/聚集，单应真值不变
SPREADS = {
    '铺满整板':        BOARD,
    '聚集在一角(1/12)': BOARD * (1/12) + 0.01,
    '近共线(窄带)':     np.column_stack([BOARD[:, 0], BOARD[:, 1] * 0.02]),
}
print(f"{'点分布':20s} {'dtype':>9s} {'未归一化':>12s} {'归一化':>12s} {'谁更好':>8s}")
for name, src in SPREADS.items():
    uvt = apply_H(H_TRUE, src)
    for dt, dn in [(np.float64, 'float64'), (np.float32, 'float32')]:
        e_no  = rel_err(src, uvt, 0.2, False, dt)
        e_yes = rel_err(src, uvt, 0.2, True,  dt)
        who = '归一化' if e_yes < e_no else '未归一化'
        print(f'{name:20s} {dn:>9s} {e_no:12.3e} {e_yes:12.3e} {who:>8s}')

print('\n✅ 三种点分布 × 两种精度，归一化都没有可测的精度收益'
      '（0.2 px 噪声引入的 ~1e-3 误差把一切数值差异都盖住了）')

## 4 · 从多视角单应解出 K（Zhang 的闭式解）

In [ ]:
def solve_K_from_homographies(views):
    '''由多视角的单应解 omega，再闭式反解 K。'''
    A = constraint_matrix(views)
    _, _, Vt = np.linalg.svd(A)
    b = Vt[-1]
    B = np.array([[b[0], b[1], b[3]],
                  [b[1], b[2], b[4]],
                  [b[3], b[4], b[5]]])
    v0  = (B[0,1]*B[0,2] - B[0,0]*B[1,2]) / (B[0,0]*B[1,1] - B[0,1]**2)
    lam = B[2,2] - (B[0,2]**2 + v0*(B[0,1]*B[0,2] - B[0,0]*B[1,2])) / B[0,0]
    al  = np.sqrt(lam / B[0,0])
    be  = np.sqrt(lam * B[0,0] / (B[0,0]*B[1,1] - B[0,1]**2))
    ga  = -B[0,1] * al**2 * be / lam
    u0  = ga * v0 / be - B[0,2] * al**2 / lam
    return np.array([[al, ga, u0], [0, be, v0], [0, 0, 1.]])

K_est = solve_K_from_homographies(VIEWS_GOOD)
print('恢复的 K：')
print(np.round(K_est, 6))
print(f'\n  f_x 相对误差 {abs(K_est[0,0]-F)/F:.3e}')
print(f'  f_y 相对误差 {abs(K_est[1,1]-F)/F:.3e}')
print(f'  c_x 相对误差 {abs(K_est[0,2]-CX)/CX:.3e}')
print(f'  c_y 相对误差 {abs(K_est[1,2]-CY)/CY:.3e}')
print(f'  skew = {K_est[0,1]:.3e}（真值 0）')

worst = max(abs(K_est[0,0]-F)/F, abs(K_est[1,1]-F)/F,
            abs(K_est[0,2]-CX)/CX, abs(K_est[1,2]-CY)/CY)
assert worst < 1e-12, f'无噪声时应精确恢复，实测 {worst:.2e}'
assert abs(K_est[0,1]) < 1e-6, 'skew 应为 0'
print(f'\n✅ 无噪声时闭式解精确恢复内参（最大相对误差 {worst:.1e}）')

# 三视角同样够
K3 = solve_K_from_homographies(VIEWS_GOOD[:3])
assert abs(K3[0,0]-F)/F < 1e-10
print('✅ 只用 3 个视角同样精确 —— 与第 2 节的 rank 计数一致')

## 5 · 重投影误差：分位数与分层

**只报均值会把离群点和边缘问题都平均掉。**

In [ ]:
def reproj_errors(Hm, src, uv_obs):
    return np.linalg.norm(apply_H(Hm, src) - uv_obs, axis=1)

rng = np.random.default_rng(5)
uv_obs = UV_TRUE + rng.normal(0, 0.25, UV_TRUE.shape)
# 注入 3 个离群点（角点检测失手）
bad_idx = rng.choice(len(uv_obs), 3, replace=False)
uv_obs[bad_idx] += rng.normal(0, 6.0, (3, 2))

H_fit, _ = dlt_homography(BOARD, uv_obs)
e = reproj_errors(H_fit, BOARD, uv_obs)

print(f'  角点数        {len(e)}')
print(f'  均值          {e.mean():.3f} px')
print(f'  中位数        {np.median(e):.3f} px')
print(f'  P95           {np.percentile(e,95):.3f} px')
print(f'  最大值        {e.max():.3f} px')
print(f'  P95/中位数    {np.percentile(e,95)/np.median(e):.1f}x')

assert np.percentile(e,95) / np.median(e) > 2.0, '离群点应当把 P95 顶起来'
assert e.mean() < np.percentile(e,95), '均值被大多数好点拉低'
print('\n✅ 3 个离群点让 P95 是中位数的数倍，而均值几乎没动'
      ' → **门禁必须用 P95**')

# 按半径分层 —— 用**所有视角的并集**，因为覆盖是整批采集的性质
print(f"\n{'配置':>16s} {'图像半径分档':>12s} {'点数':>6s}")
for cfg_name, uv_all in [('VIEWS_GOOD (5)', UV_ALL_GOOD), ('VIEWS_FULL (9)', UV_ALL_FULL)]:
    rn = np.linalg.norm(uv_all - np.array([CX, CY]), axis=1) / RD
    for lo, hi, nm in [(0.0, 1/3, '内 1/3'), (1/3, 2/3, '中 1/3'), (2/3, 9.9, '外 1/3')]:
        cnt = int(((rn >= lo) & (rn < hi)).sum())
        print(f'{cfg_name:>16s} {nm:>12s} {cnt:6d}')
    print()

rn_g = np.linalg.norm(UV_ALL_GOOD - [CX, CY], axis=1) / RD
rn_f = np.linalg.norm(UV_ALL_FULL - [CX, CY], axis=1) / RD
assert (rn_g >= 2/3).sum() <= 2, 'VIEWS_GOOD 在外圈几乎没有点'
assert (rn_f >= 2/3).sum() > 30, 'VIEWS_FULL 才真正覆盖了外圈'
print('✅ 分层报告暴露了一件 rank 检查看不到的事：'
      'VIEWS_GOOD 的外圈只有 1 个角点')
print('   → 用它标出来的畸变模型在画幅角上是**外推**，'
      '而模块 01 量过那里的位移是 197.71 px')

## 6 · 可辨识性：想标好远处，必须采近处

$\partial v/\partial c_y = 1$，而
$\partial v/\partial\text{pitch} = f\sec^2\theta \approx f\bigl(1+((v-c_y)/f)^2\bigr)$。
**第二列只在 $v$ 跨度大时才与第一列分开。**

In [ ]:
def jac_cy_pitch(vs):
    '''(c_y, pitch[rad]) 对像素行 v 的雅可比。'''
    vs = np.asarray(vs, float)
    return np.column_stack([np.ones_like(vs), F * (1 + ((vs - CY) / F) ** 2)])

def identifiability(vs, corner_noise_px=0.2):
    J = jac_cy_pitch(vs)
    cov = np.linalg.inv(J.T @ J) * corner_noise_px ** 2
    return {'delta_v': float(np.ptp(vs)),
            'cond': float(np.linalg.cond(J)),
            'sigma_cy_px': float(np.sqrt(cov[0, 0])),
            'sigma_pitch_deg': float(np.rad2deg(np.sqrt(cov[1, 1])))}

COVERAGE = {
    '只用远处地面 (v 570–600)':      np.linspace(570, 600, 20),
    '中等范围 (v 570–720)':          np.linspace(570, 720, 20),
    '全画幅地面 (v 570–1070)':       np.linspace(570, 1070, 20),
    '再加地平线以上 (v 380–1070)':   np.linspace(380, 1070, 30),
}
print(f"{'采集覆盖':30s} {'Δv':>6s} {'cond':>11s} {'σ(c_y)':>9s} {'σ(pitch)':>10s}")
ident = {}
for name, vs in COVERAGE.items():
    r = identifiability(vs)
    ident[name] = r
    print(f"{name:30s} {r['delta_v']:6.0f} {r['cond']:11.3e} "
          f"{r['sigma_cy_px']:8.2f}px {r['sigma_pitch_deg']:9.4f}°")

far  = ident['只用远处地面 (v 570–600)']['sigma_pitch_deg']
full = ident['全画幅地面 (v 570–1070)']['sigma_pitch_deg']
print(f'\n只采远处 σ(pitch) = {far:.2f}°   全画幅 σ(pitch) = {full:.4f}°'
      f'   → 改善 **{far/full:.0f} 倍**')
assert far > 3.0, '只采远处时 σ(pitch) 应超过 3°'
assert full < 0.05, '全画幅覆盖应把 σ(pitch) 压到 0.05° 以下'
assert far / full > 50, f'改善应超过 50 倍，实测 {far/full:.0f}'

# 收益会饱和
sat = ident['再加地平线以上 (v 380–1070)']['sigma_pitch_deg']
print(f'再扩到 Δv=690: σ(pitch) = {sat:.4f}°  (仅再降 {100*(1-sat/full):.0f}%)')
assert sat / full > 0.8, '继续扩大覆盖的收益应当饱和'

# 与下游需求挂钩：50m 处 2m 误差要求 pitch 多准？
def pitch_err_to_m(deg, d=50.0):
    th = np.arctan2(H_CAM, d)
    th2 = th - np.deg2rad(deg)
    return abs(H_CAM / np.tan(th2) - d) if th2 > 1e-9 else np.inf
for need in [2.0]:
    lo, hi = 1e-4, 2.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if pitch_err_to_m(mid) < need: lo = mid
        else: hi = mid
    print(f'\n若要求 50m 处误差 < {need}m，则 pitch 误差必须 < **{lo:.4f}°**')
    print(f'  只采远处 ({far:.2f}°) 差 {far/lo:.0f} 倍；'
          f'全画幅 ({full:.4f}°) {"够用" if full < lo else f"仍差 {full/lo:.1f} 倍"}')
print('\n✅ 最方便的采集方式（只对着远处拍）是最差的选择')

## 7 · 小结

| 结论 | 数值 |
|---|---|
| Zhang 的约束计数 | 1/2/3 视角 → rank 2/4/**5**，在 5 饱和 |
| 退化采集 | 纯平移、只绕光轴转 → **拍 5 张仍是 rank 2** |
| 只倾斜一个轴 | rank 5，**够用** |
| 归一化（无噪声） | 精度提升 **数百倍** |
| 归一化（0.2 px 噪声） | **差别消失**，两者都 ~1e-3 |
| 闭式解恢复 K | 无噪声下相对误差 < 1e-12 |
| 离群点 | 3 个点让 P95/中位数 > 2×，均值几乎不动 |
| **可辨识性** | σ(pitch) **3.74° → 0.035°**，靠扩大 Δv 而非多拍 |

## ✏️ 练习 1：带归一化的 DLT

实现 `homography_dlt(src, dst, normalize=True)`，返回 `(H, cond(A))`，
其中 `H[2,2] == 1`。要求归一化用「质心 + 平均距离 $\sqrt2$」的标准做法。

In [ ]:
def homography_dlt(src, dst, normalize=True):
    """返回 (H, cond(A))；H 归一化到 H[2,2]=1。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
SRC = BOARD
DST = apply_H(H_TRUE, SRC)

for nz in [False, True]:
    Hm, c = homography_dlt(SRC, DST, normalize=nz)
    assert abs(Hm[2, 2] - 1.0) < 1e-12, 'H 应归一化到 H[2,2]=1'
    err = np.linalg.norm(Hm - H_TRUE) / np.linalg.norm(H_TRUE)
    print(f'normalize={str(nz):5s}  cond={c:11.3e}  相对误差={err:.3e}')
    assert err < 1e-8, f'无噪声时应精确，实测 {err:.2e}'

c_no  = homography_dlt(SRC, DST, normalize=False)[1]
c_yes = homography_dlt(SRC, DST, normalize=True)[1]
assert c_no / c_yes > 1e3, f'归一化应把条件数改善 3 个数量级以上（实测 {c_no/c_yes:.1e}）'

# 少于 4 点应当无法确定单应
try:
    homography_dlt(SRC[:3], DST[:3])
    got = True
except Exception:
    got = False
print(f'\n3 个点时{"没有报错（解不唯一，但 SVD 仍会给一个）" if got else "抛出异常"}')
print('✅ 练习 1 通过：无噪声下精确，且归一化把条件数改善 3+ 个数量级')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def homography_dlt(src, dst, normalize=True):
    src = np.asarray(src, float); dst = np.asarray(dst, float)
    if normalize:
        s_n, Ts = normalize_pts(src); d_n, Td = normalize_pts(dst)
    else:
        s_n, d_n, Ts, Td = src, dst, np.eye(3), np.eye(3)
    A = []
    for (x, y), (u, v) in zip(s_n, d_n):
        A.append([-x, -y, -1,  0,  0,  0, u*x, u*y, u])
        A.append([ 0,  0,  0, -x, -y, -1, v*x, v*y, v])
    A = np.array(A)
    cond = float(np.linalg.cond(A))
    _, _, Vt = np.linalg.svd(A)
    Hm = np.linalg.inv(Td) @ Vt[-1].reshape(3, 3) @ Ts
    return Hm / Hm[2, 2], cond

Hm, c = homography_dlt(BOARD, apply_H(H_TRUE, BOARD))
assert np.linalg.norm(Hm - H_TRUE) / np.linalg.norm(H_TRUE) < 1e-8
print('✅ 参考答案 2 通过'.replace('2', '1'))

## ✏️ 练习 2：采集配置审计

实现 `collection_audit(views)`，在**优化之前**就判断这批图能不能标：

- `'n_views'` —— 视角数
- `'rank'` —— 约束矩阵的 rank
- `'solvable'` —— bool：`rank >= 5`
- `'degenerate_reason'` —— str 或 None：`rank < 5` 时给出原因
  （`'too_few_views'` 若视角数 < 3，否则 `'insufficient_orientation_diversity'`）

In [ ]:
def collection_audit(views):
    """返回 dict(n_views, rank, solvable, degenerate_reason)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
CASES = {
    '3 视角（不同朝向）':   VIEWS_GOOD[:3],
    '5 视角（不同朝向）':   VIEWS_GOOD,
    '2 视角':               VIEWS_GOOD[:2],
    '退化：5 张纯平移':     CONFIGS['退化：5 张纯平移'],
    '退化：5 张绕光轴转':   CONFIGS['退化：5 张只绕光轴转'],
}
for name, vs in CASES.items():
    a = collection_audit(vs)
    assert set(a) == {'n_views', 'rank', 'solvable', 'degenerate_reason'}
    print(f"{name:22s} n={a['n_views']} rank={a['rank']} "
          f"solvable={str(a['solvable']):5s} reason={a['degenerate_reason']}")

assert collection_audit(VIEWS_GOOD[:3])['solvable'] is True
assert collection_audit(VIEWS_GOOD[:3])['degenerate_reason'] is None
a2 = collection_audit(VIEWS_GOOD[:2])
assert a2['solvable'] is False and a2['degenerate_reason'] == 'too_few_views'
for k in ['退化：5 张纯平移', '退化：5 张绕光轴转']:
    a = collection_audit(CASES[k])
    assert a['solvable'] is False
    assert a['degenerate_reason'] == 'insufficient_orientation_diversity', a
    assert a['n_views'] == 5, '视角数够，但朝向多样性不够 —— 两个信号必须分开'
print('\n✅ 练习 2 通过：「张数够」与「朝向够」是两个独立的判据')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def collection_audit(views):
    n = len(views)
    A = constraint_matrix(views)
    r = int(np.linalg.matrix_rank(A, tol=1e-8))
    ok = r >= 5
    reason = None
    if not ok:
        reason = 'too_few_views' if n < 3 else 'insufficient_orientation_diversity'
    return {'n_views': n, 'rank': r, 'solvable': ok, 'degenerate_reason': reason}

assert collection_audit(VIEWS_GOOD)['solvable']
assert collection_audit(CONFIGS['退化：5 张纯平移'])['degenerate_reason'] \
       == 'insufficient_orientation_diversity'
print('✅ 参考答案 2 通过')

## ✏️ 练习 3：可解读的重投影报告

实现 `reproj_report(errs, uv)`，返回 dict：

- `'n'`, `'median'`, `'p95'`, `'max'`
- `'p95_outer'` —— 图像外 1/3 半径区域的 P95（无点时 `float('nan')`）
- `'outlier_ratio'` —— 误差大于 `5 × 中位数` 的点占比

> 为什么是 5 倍而不是 3 倍：误差近似半正态时中位数 $=0.6745\sigma$，
> 所以 `3×中位数` 只有 $2.02\sigma$ —— **干净数据上就有 4.3% 的假阳性**
> （实测 4.269%，理论 4.302%）。`5×中位数` 是 $3.37\sigma$，假阳性 **0.07%**。
- `'coverage_frac'` —— 角点包围盒面积 / 图像面积
- `'n_outer'` —— 落在外 1/3 半径区域的角点数

> `uv` 应当传**所有视角的并集**——覆盖是整批采集的性质，不是单张图的性质。

In [ ]:
def reproj_report(errs, uv, img_w=W, img_h=HGT):
    """返回 dict(n, median, p95, max, p95_outer, outlier_ratio, coverage_frac)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# —— 所有 fixture 先建好（放在调用 stub 之前）——
rng2 = np.random.default_rng(5)

UV_ONE  = apply_H(H_of_view(*VIEWS_GOOD[1]), BOARD)          # 单视角 54 点
e_one   = np.abs(rng2.normal(0, 0.25, len(UV_ONE)))
e_one3  = e_one.copy()
e_one3[rng2.choice(len(e_one), 3, replace=False)] += 6.0      # 3/54 = 5.6%

e_good  = np.abs(rng2.normal(0, 0.25, len(UV_ALL_GOOD)))      # 5 视角并集
e_full  = np.abs(rng2.normal(0, 0.25, len(UV_ALL_FULL)))      # 9 视角并集
e_few   = e_full.copy()
e_few[rng2.choice(len(e_full), 6, replace=False)] += 6.0       # 6/486 = 1.2%
e_dirty = e_full.copy()
e_dirty[rng2.choice(len(e_full), 20, replace=False)] += 6.0    # 20/486 = 4.1%

uv_small = (UV_ALL_FULL - [CX, CY]) * 0.25 + [CX, CY]

# —— 第一组：离群点占比 5.6% 时，P95 抓得住 ——
r_one, r_one3 = reproj_report(e_one, UV_ONE), reproj_report(e_one3, UV_ONE)
for r in (r_one, r_one3):
    assert set(r) == {'n', 'median', 'p95', 'max', 'p95_outer',
                      'outlier_ratio', 'coverage_frac', 'n_outer'}
print(f"54 点 + 3 个离群点 (5.6%)：P95 {r_one['p95']:.3f} -> {r_one3['p95']:.3f}，"
      f"均值 {e_one.mean():.3f} -> {e_one3.mean():.3f}")
assert r_one3['p95'] > r_one['p95'] * 2, '占比 5.6% 时 P95 必须被顶起来'

# —— 第二组：占比 1.2% 时 P95 看不见 ——
r_full_r, r_few = reproj_report(e_full, UV_ALL_FULL), reproj_report(e_few, UV_ALL_FULL)
print(f"486 点 + 6 个离群点 (1.2%)：P95 {r_full_r['p95']:.3f} -> {r_few['p95']:.3f}，"
      f"outlier_ratio {r_full_r['outlier_ratio']:.4f} -> {r_few['outlier_ratio']:.4f}")
assert r_few['p95'] < r_full_r['p95'] * 1.2, '占比 1.2% 时 P95 几乎不动'
assert r_few['outlier_ratio'] > 0.005 and r_full_r['outlier_ratio'] < 0.005,     'outlier_ratio 才抓得住（干净数据上它应当接近 0）'
print('  -> **P95 的检出能力取决于离群点占比**：低于 5% 时它看不见，'
      '必须同时报 outlier_ratio')

# —— 第三组：覆盖是整批采集的性质 ——
r_g = reproj_report(e_good, UV_ALL_GOOD)
print(f"\nGOOD(5): coverage={r_g['coverage_frac']:.3f} n_outer={r_g['n_outer']}")
print(f"FULL(9): coverage={r_full_r['coverage_frac']:.3f} n_outer={r_full_r['n_outer']}")
assert r_full_r['coverage_frac'] > r_g['coverage_frac'] * 2
assert r_g['n_outer'] <= 2 and r_full_r['n_outer'] > 30
print(f"GOOD(5) 的 p95_outer = {r_g['p95_outer']}"
      f"  (外圈只有 {r_g['n_outer']} 个点 -> 统计量无意义)")
assert r_full_r['p95_outer'] == r_full_r['p95_outer'], 'FULL 才有真实的外圈 P95'

# —— 第四组：覆盖率抓「板拍太远」——
assert reproj_report(e_full, uv_small)['coverage_frac'] < r_full_r['coverage_frac'] / 10
print('\n✅ 练习 3 通过：P95 与 outlier_ratio 分工检出离群点、'
      'n_outer 抓边缘覆盖、coverage_frac 抓「板拍太远」')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def reproj_report(errs, uv, img_w=W, img_h=HGT):
    e = np.asarray(errs, float); uv = np.asarray(uv, float)
    med = float(np.median(e))
    r_norm = np.linalg.norm(uv - np.array([img_w/2, img_h/2]), axis=1) \
             / np.hypot(img_w/2, img_h/2)
    outer = e[r_norm >= 2/3]
    bbox = (np.ptp(uv[:, 0]) * np.ptp(uv[:, 1])) / (img_w * img_h)   # numpy 2 移除了 ndarray.ptp()
    return {'n': int(len(e)),
            'median': med,
            'p95': float(np.percentile(e, 95)),
            'max': float(e.max()),
            'p95_outer': float(np.percentile(outer, 95)) if len(outer) else float('nan'),
            'outlier_ratio': float((e > 5 * med).mean()),   # 5 倍：假阳性 0.07%
            'coverage_frac': float(min(bbox, 1.0)),
            'n_outer': int(len(outer))}

r = reproj_report(e_one3, UV_ONE)
assert r['p95'] > r['median'] * 2
assert reproj_report(e_good, UV_ALL_GOOD)['n_outer'] <= 2
print('✅ 参考答案 3 通过')
print('   coverage_frac 用包围盒（抓「板太远」与「板只在画面一角」）；'
      'n_outer 单独一项，因为覆盖率高不代表覆盖到了**边缘**。')

## ✏️ 练习 4：标定验收器

把本模块所有判据装进一个函数。实现
`calib_acceptance(views, errs, uv, v_rows)`，返回
`(是否通过, {检查项: (通过?, 实测值)})`，包含：

| 检查项 | 阈值 |
|---|---|
| `zhang_rank` | `== 5` |
| `reproj_p95` | `< 0.8` px |
| `reproj_p95_outer` | `< 1.2` px |
| `outlier_ratio` | `< 0.02` |
| `coverage_frac` | `>= 0.15` |
| `n_outer` | `>= 20` |
| `delta_v` | `>= 400` px |
| `sigma_pitch_deg` | `< 0.05` |

**只有全部通过才返回 True。**

> 自测里会出现一个值得注意的结果：**`VIEWS_GOOD`（5 视角）通不过**——
> 它的 rank、Δv、重投影误差全都合格，栽在 `n_outer` 上。

In [ ]:
def calib_acceptance(views, errs, uv, v_rows):
    """返回 (bool, {检查项: (通过?, 实测值)})。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
V_WIDE = np.linspace(570, 1070, 40)     # 全画幅地面覆盖
V_FAR  = np.linspace(570, 600,  40)     # 只采远处

def show(tag, res):
    ok, det = res
    print(f'{tag}  ->  ' + ('通过' if ok else '**不通过**'))
    for k, (passed, val) in det.items():
        v = f'{val:.4f}' if isinstance(val, float) else str(val)
        print(f"   {'OK  ' if passed else 'FAIL'} {k:20s} = {v}")
    print()

r_full_acc = calib_acceptance(VIEWS_FULL, e_full, UV_ALL_FULL, V_WIDE)
show('VIEWS_FULL (9 视角) + 全画幅 Δv', r_full_acc)
assert r_full_acc[0] is True, r_full_acc[1]

# ★ 本模块最有意思的一条：五视角配置栽在边缘覆盖上
r_good_acc = calib_acceptance(VIEWS_GOOD, e_good, UV_ALL_GOOD, V_WIDE)
show('VIEWS_GOOD (5 视角) + 全画幅 Δv', r_good_acc)
assert r_good_acc[0] is False, '五视角配置不该通过'
assert r_good_acc[1]['zhang_rank'][0] is True, 'rank 是合格的'
assert r_good_acc[1]['delta_v'][0] is True, 'Δv 是合格的'
assert r_good_acc[1]['reproj_p95'][0] is True, '重投影误差是合格的'
assert r_good_acc[1]['n_outer'][0] is False, '**它只栽在 n_outer 上**'

# 退化采集
bad1 = calib_acceptance(CONFIGS['退化：5 张纯平移'], e_good, UV_ALL_GOOD, V_WIDE)
assert bad1[0] is False and bad1[1]['zhang_rank'][0] is False

# 离群点
bad2 = calib_acceptance(VIEWS_FULL, e_dirty, UV_ALL_FULL, V_WIDE)
assert bad2[0] is False
assert (bad2[1]['outlier_ratio'][0] is False) or (bad2[1]['reproj_p95'][0] is False)

# 只采远处：Δv 与 σ(pitch) 同时报警
bad3 = calib_acceptance(VIEWS_FULL, e_full, UV_ALL_FULL, V_FAR)
assert bad3[0] is False
assert bad3[1]['delta_v'][0] is False and bad3[1]['sigma_pitch_deg'][0] is False,     '只采远处应当同时触发 Δv 与 σ(pitch) 两项'
print(f"只采远处时 σ(pitch) = {bad3[1]['sigma_pitch_deg'][1]:.2f}°"
      '  → **两项同时报警，因为它们量的是同一件事**')
print()
print('✅ 练习 4 通过。而最值得记的是第二组：'
      '**本课自己的 5 视角配置通不过验收**——')
print('   rank / Δv / 重投影误差全合格，栽在「一个角点都没采到边缘」上。')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def calib_acceptance(views, errs, uv, v_rows):
    a = collection_audit(views)
    r = reproj_report(errs, uv)
    idt = identifiability(v_rows)
    checks = {
        'zhang_rank':       (a['rank'] == 5,                a['rank']),
        'reproj_p95':       (r['p95'] < 0.8,                r['p95']),
        'reproj_p95_outer': (not (r['p95_outer'] >= 1.2),   r['p95_outer']),
        'outlier_ratio':    (r['outlier_ratio'] < 0.02,     r['outlier_ratio']),
        'coverage_frac':    (r['coverage_frac'] >= 0.15,    r['coverage_frac']),
        'n_outer':          (r['n_outer'] >= 20,            r['n_outer']),
        'delta_v':          (idt['delta_v'] >= 400,         idt['delta_v']),
        'sigma_pitch_deg':  (idt['sigma_pitch_deg'] < 0.05, idt['sigma_pitch_deg']),
    }
    return all(p for p, _ in checks.values()), checks

assert calib_acceptance(VIEWS_FULL, e_full, UV_ALL_FULL,
                        np.linspace(570, 1070, 40))[0] is True
assert calib_acceptance(VIEWS_GOOD, e_good, UV_ALL_GOOD,
                        np.linspace(570, 1070, 40))[0] is False
print('✅ 参考答案 4 通过')
print('   两个实现细节：')
print('   ① p95_outer 用 `not (x >= 1.2)` 而不是 `x < 1.2` —— '
      '外圈无点时它是 nan，而 `nan < 1.2` 是 False，会把「没采到边缘」误判成 FAIL；')
print('   ② 真正抓「没采到边缘」的是 n_outer，'
      '它是**计数**而不是误差，所以不会被 nan 干扰。')

## 🧪 真实工程胶囊

```python
# ── 1) OpenCV 的标定入口 ──
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, (w, h), None, None,
    flags=cv2.CALIB_ZERO_TANGENT_DIST |      # 先不标 p1,p2（模块 01 第 3 节）
          cv2.CALIB_FIX_K3)                  # k3 只在广角上需要
#   ⚠️ ret 是**均值** RMS —— 它就是第 5 节说的「只报均值」。
#      要自己算 P95：
per_pt = [np.linalg.norm(cv2.projectPoints(o, r, t, K, dist)[0].reshape(-1,2) - i,
                         axis=1) for o, i, r, t in zip(objpoints, imgpoints, rvecs, tvecs)]
e = np.concatenate(per_pt)
report = dict(median=np.median(e), p95=np.percentile(e, 95), max=e.max())

# ── 2) 采集现场就能跑的两条检查（练习 2 与第 6 节）──
#    都不需要跑优化，拍完立刻算：
audit = collection_audit(homographies)          # rank 够不够
assert audit['solvable'], audit['degenerate_reason']
assert np.ptp(all_corner_rows) >= 400, '角点的图像行跨度不足，pitch 标不准'

# ── 3) 外参：写进 ROS 的 TF 静态变换，而不是散落在代码里 ──
# extrinsics.yaml
#   parent_frame: base_link      # REP-105
#   child_frame:  camera_front
#   translation:  [1.70, 0.00, 1.50]     # ← Z=0 指哪个平面必须在文档里写明
#   rotation_rpy: [0.0, 0.0342, 0.0]     # 弧度
#   load_state:   empty                  # ← 第 5b 节：载荷状态影响 H，值 7.4%
#   sigma_pitch_deg: 0.034               # ← 由练习 4 反解，供下游算 sigma_m
```

> **落地顺序建议**：先加采集现场的两条检查（零成本、抓一整类失败），
> 再把重投影误差从「均值」改成「中位数/P95/外圈 P95」三个数，
> 最后才是把 σ(pitch) 接到下游的 `sigma_m` 上。